# Stage 7: public pretrained OOF residual gate

This standalone Colab notebook reconstructs the ravaghi public-stack OOF predictions, cross-fits a small residual corrector, and decides whether it is safe to spend a Kaggle run. It does not create a submission.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
python = repo_dir / '.venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'kagglehub', 'lightgbm', 'catboost'], check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print('Repository and Python environment are ready.')

In [ ]:
# Download the public pretrained artifact. This is done in Colab, where Internet is ON.
download = subprocess.run([
    str(python), '-c',
    "import kagglehub; print(kagglehub.dataset_download('ravaghi/wellbore-geology-prediction-artifacts'))"
], check=True, capture_output=True, text=True)
print(download.stdout)
download_root = Path(download.stdout.strip().splitlines()[-1])
train_candidates = list(download_root.rglob('data/train.csv'))
assert len(train_candidates) == 1, f'Expected one public train.csv, found: {train_candidates}'
public_artifacts_dir = train_candidates[0].parent.parent
print('Public artifacts:', public_artifacts_dir)

In [ ]:
# koolbox is needed only to deserialize the five public Trainer objects.
probe = subprocess.run([str(python), '-c', 'import koolbox'], capture_output=True)
if probe.returncode != 0:
    direct = subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'koolbox==0.1.3'])
    if direct.returncode != 0:
        kb_download = subprocess.run([
            str(python), '-c',
            "import kagglehub; print(kagglehub.dataset_download('phongnguyn23021656/koolbox-offline'))"
        ], check=True, capture_output=True, text=True)
        kb_root = Path(kb_download.stdout.strip().splitlines()[-1])
        wheels = list(kb_root.rglob('*.whl'))
        assert wheels, f'No koolbox wheel found under {kb_root}'
        for wheel in wheels:
            subprocess.run(['uv', 'pip', 'install', '--python', str(python), '--no-deps', str(wheel)], check=True)
subprocess.run([str(python), '-c', 'import koolbox; print(koolbox.__file__)'], check=True)

## Run the full OOF gate

Leave `LIMIT_WELLS = None` for the decision-making run. A small value is only a wiring smoke test and must never be used to approve a Kaggle submission.

In [ ]:
RUN_ID = 'stage7_public_residual_gate_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-public-oof',
        '--config', 'configs/experiment/public_residual_gate.yaml',
        '--public-artifacts-dir', str(public_artifacts_dir),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    subprocess.run(command, cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)

In [ ]:
gate = json.loads((run_dir / 'gate_summary.json').read_text())
weights = json.loads((run_dir / 'weight_grid.json').read_text())
{
    'promoted': gate['promoted'],
    'base_rmse': gate['base_metrics']['pooled_rmse'],
    'candidate_rmse': gate['candidate_metrics']['pooled_rmse'],
    'rmse_delta': gate['pooled_rmse_delta'],
    'bootstrap_95pct': [gate['bootstrap']['ci_2_5'], gate['bootstrap']['ci_97_5']],
    'improved_folds': f"{gate['improved_folds']}/{len(gate['fold_deltas'])}",
    'gates': gate['gates'],
    'spatial_delta': gate['spatial']['pooled_rmse_delta'],
    'weight_grid': weights,
}

## Decision

Proceed to a Kaggle inference notebook only when `promoted` is `True`. The gate requires a useful pooled gain, improvement in at least 4/5 normal folds and 5/6 spatial folds, a paired-well bootstrap upper bound below zero, and no deterioration in P90 or worst-well concentration. If it is false, send the final dictionary back for diagnosis; do not submit.